In [1]:
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import Descriptors
from skfp.fingerprints import PubChemFingerprint
from mordred import Calculator, descriptors
from sklearn.svm import SVR
import joblib

def make_Tc_feature(smiles, names):
    fea_rdkit = ['MinAbsEStateIndex','MinEStateIndex','qed','HeavyAtomMolWt','MaxPartialCharge','MinPartialCharge','MaxAbsPartialCharge','MinAbsPartialCharge','BCUT2D_MWHI','BCUT2D_CHGLO','BCUT2D_LOGPLOW','BCUT2D_MRHI','BCUT2D_MRLOW','BertzCT','Chi1v','Chi3v','HallKierAlpha','Kappa2','LabuteASA','TPSA','VSA_EState1','VSA_EState3','VSA_EState8','NHOHCount','NumHeteroatoms','MolLogP','MolMR']
    rdkit2D = {}
    pubchem_fp = PubChemFingerprint()
    for i in range(len(smiles)):
        name = names[i]
        smile = smiles[i]
        mol = Chem.MolFromSmiles(smile)
        rdkit2D[name] = {}
        rdkit2D[name]['smiles'] = smile
        # MaxAbsEStateIndex
        for Descriptor, func in Descriptors.descList:
            if Descriptor in fea_rdkit:
                try:
                    rdkit2D[name][Descriptor] = func(mol)
                except Exception as e:
                    rdkit2D[name][Descriptor] = 0
                    print(f"Error calculating {Descriptor} for {name}: {e}")
                    pass
    calc = Calculator(descriptors, ignore_3D=False)
    mols = [Chem.MolFromSmiles(smile) for smile in smiles]
    mordred = calc.pandas(mols, quiet=True)
    # 只保留下面这些特征
    fea = ['SpMAD_A','ATS2dv','ATS3dv','ATS4dv','ATS0Z','ATS1Z','ATS0m','ATS1m','ATS2m','ATS3m','ATS0p','ATS1p','ATS2p','AATS0dv','AATS0se','AATS0are','AATS0i','ATSC0dv','ATSC1dv','ATSC0Z','ATSC0m','ATSC0v','ATSC1v','ATSC0se','ATSC1se','ATSC0pe','ATSC1pe','ATSC0are','ATSC1are','ATSC1p','ATSC1i','AATSC0dv','AATSC0m','AATSC0pe','AATSC0are','AATSC0i','BCUTdv-1h','BCUTdv-1l','BCUTare-1h','BCUTp-1l','BCUTi-1h','SpAbs_Dzp','SM1_Dzp','VR2_Dzp','SM1_Dzi','Xp-5d','Xp-6d','Xp-7d','Xp-1dv','SZ','Sp','Mse','Mare','Mi','SpMAD_Dt','SsCH3','ETA_alpha','AETA_alpha','ETA_beta','ETA_beta_s','AETA_beta_s','ETA_epsilon_2','ETA_epsilon_5','ETA_dEpsilon_D','fragCpx','IC0','IC1','TIC0','TIC1','TIC2','TIC4','MIC0','MIC2','MIC4','MIC5','ZMIC0','ZMIC1','ZMIC2','FilterItLogS','VMcGowan','AMID','MID_h','MPC7','MPC10','piPC1','piPC2','piPC5','piPC6','piPC10','TpiPC10','apol','TopoPSA']
    mordred = mordred[fea]
    df_ = pd.DataFrame(rdkit2D).T
    # df_转化为表格格式，不需要index
    df_ = df_.reset_index()
    # 重命名列
    df_.columns = ['Name', 'smiles','MinAbsEStateIndex','MinEStateIndex','qed','HeavyAtomMolWt','MaxPartialCharge','MinPartialCharge','MaxAbsPartialCharge','MinAbsPartialCharge','BCUT2D_MWHI','BCUT2D_CHGLO','BCUT2D_LOGPLOW','BCUT2D_MRHI','BCUT2D_MRLOW','BertzCT','Chi1v','Chi3v','HallKierAlpha','Kappa2','LabuteASA','TPSA','VSA_EState1','VSA_EState3','VSA_EState8','NHOHCount','NumHeteroatoms','MolLogP','MolMR']
    df_ = pd.concat([df_, mordred], axis=1)
    # df_.to_csv('./exp/Tc_feature.csv', index=False)
    return df_

def predict_Tc(smiles, names = None):
    svr = joblib.load('./model/RF-Tc-2D.pkl')
    mean = pd.read_csv('./data/Tc_mean.csv', header=None)
    std = pd.read_csv('./data/Tc_std.csv', header=None)
    mean = mean.values[:119].reshape(-1)
    std = std.values[:119].reshape(-1)
    if names == None:
        names = smiles
    make_Tc_feature_ = make_Tc_feature(smiles, names)
    # make_Tc_feature_.to_excel('./exp/TC_feature.xlsx', index=False)
    make_Tc_feature_ = make_Tc_feature_.drop(columns=['Name', 'smiles'])
    # 归一化
    make_Tc_feature_ = make_Tc_feature_.values
    make_Tc_feature_ = (make_Tc_feature_ - mean) / std
    # 预测
    Tc = svr.predict(make_Tc_feature_)
    return Tc

In [2]:
# 打开exp/Critical.xlsx
df = pd.read_excel('./newexp/TEST_A.xlsx')
# 重设索引
df = df.reset_index(drop=True)
smiles = df['smiles'].tolist()
pred_TC = predict_Tc(smiles)
df['TC_pred'] = np.exp(pred_TC)
df['AARD(%)'] = 100 * abs(df['TC_pred'] - df['Tc (K)']) / df['Tc (K)']
# 保存到文件
df.to_excel('./newexp/RF_TC_2D_TEST_A.xlsx', index=False)
from sklearn.metrics import mean_squared_error, r2_score
Tc_pred = df['TC_pred'].values
TC = df['Tc (K)'].values
tc_r2 = r2_score(TC, Tc_pred)
tc_rmse = np.sqrt(mean_squared_error(TC, Tc_pred))
tc_mae = np.mean(np.abs(TC - Tc_pred))
tc_aard = np.mean(np.abs(TC - Tc_pred) / TC) * 100
print(f'Tc R^2: {tc_r2:.4f}, RMSE: {tc_rmse:.4f}, MAE: {tc_mae:.4f}, AARD: {tc_aard:.4f}')

Tc R^2: 0.7713, RMSE: 62.8999, MAE: 33.1103, AARD: 6.3198


In [3]:
# 打开exp/Critical.xlsx
df = pd.read_excel('./newexp/TEST_AB.xlsx')
# 重设索引
df = df.reset_index(drop=True)
smiles = df['smiles'].tolist()
pred_TC = predict_Tc(smiles)
df['TC_pred'] = np.exp(pred_TC)
df['AARD(%)'] = 100 * abs(df['TC_pred'] - df['Tc (K)']) / df['Tc (K)']
# 保存到文件
df.to_excel('./newexp/RF_TC_2D_TEST_AB.xlsx', index=False)
from sklearn.metrics import mean_squared_error, r2_score
Tc_pred = df['TC_pred'].values
TC = df['Tc (K)'].values
tc_r2 = r2_score(TC, Tc_pred)
tc_rmse = np.sqrt(mean_squared_error(TC, Tc_pred))
tc_mae = np.mean(np.abs(TC - Tc_pred))
tc_aard = np.mean(np.abs(TC - Tc_pred) / TC) * 100
print(f'Tc R^2: {tc_r2:.4f}, RMSE: {tc_rmse:.4f}, MAE: {tc_mae:.4f}, AARD: {tc_aard:.4f}')

Tc R^2: 0.7265, RMSE: 72.3230, MAE: 45.6653, AARD: 8.7437


In [4]:
# 打开exp/Critical.xlsx
df = pd.read_excel('./newexp/TEST_AB.xlsx')
df2 = pd.read_excel('./newexp/TEST_A.xlsx')
# dfdrop掉df2中已有的smiles
df = df[~df['smiles'].isin(df2['smiles'])]
# 重设索引
smiles = df['smiles'].tolist()
pred_TC = predict_Tc(smiles)
df['TC_pred'] = np.exp(pred_TC)
df['AARD(%)'] = 100 * abs(df['TC_pred'] - df['Tc (K)']) / df['Tc (K)']
from sklearn.metrics import mean_squared_error, r2_score
Tc_pred = df['TC_pred'].values
TC = df['Tc (K)'].values
tc_r2 = r2_score(TC, Tc_pred)
tc_rmse = np.sqrt(mean_squared_error(TC, Tc_pred))
tc_mae = np.mean(np.abs(TC - Tc_pred))
tc_aard = np.mean(np.abs(TC - Tc_pred) / TC) * 100
print(f'Tc R^2: {tc_r2:.4f}, RMSE: {tc_rmse:.4f}, MAE: {tc_mae:.4f}, AARD: {tc_aard:.4f}')

Tc R^2: 0.5968, RMSE: 100.9622, MAE: 94.5636, AARD: 18.1840


In [5]:
df = pd.read_excel('./homo/PURE-homo-150.xlsx')
SMILES = df['SMILES'].tolist()
names = df['name'].tolist()
pred_Tc = predict_Tc(SMILES, names)
df['Tc'] = np.exp(pred_Tc)
df.to_csv('./homo/RF_2D_Tc.csv', index=False)